In [0]:
dbutils.widgets.text("destination_timestamp","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("merge_key","")
dbutils.widgets.text("load_type","")

destination_timestamp = dbutils.widgets.get("destination_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_key = dbutils.widgets.get("merge_key")
load_type = dbutils.widgets.get("load_type")


In [0]:
bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"destination_timestamp = '{destination_timestamp}'")

In [0]:
from pyspark.sql import functions as F
updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("updated_at", F.current_timestamp())

In [0]:
from delta.tables import DeltaTable

target_table  = f"adb_caresync.silver.{table_name}"
source_cols   = updated_df.columns
source_count  = updated_df.count()
table_exists  = spark.catalog.tableExists(target_table)

print("=" * 65)
print("  BRONZE → SILVER LOAD")
print("=" * 65)
print(f"  Table          : {table_name}")
print(f"  Target         : {target_table}")
print(f"  Load Type      : {load_type}")
print(f"  Merge Key      : {merge_key if merge_key else 'N/A'}")
print(f"  Source Rows    : {source_count}")
print(f"  Columns ({len(source_cols):>2})   : {source_cols}")
print(f"  Target Exists  : {table_exists}")
print("=" * 65)

# ─────────────────────────────────────────────────────────────
# FULL — overwrite entire target table
# ─────────────────────────────────────────────────────────────
if load_type == "FULL":
    print(f"\n[FULL] Strategy: Overwrite entire target table")
    print(f"\n[FULL] Equivalent SQL:")
    print(f"  INSERT OVERWRITE TABLE {target_table}")
    print(f"  SELECT {', '.join(source_cols)}")
    print(f"  FROM source_data;")
    print(f"\n[FULL] Executing overwrite...")
    updated_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
    print(f"[FULL] ✓ Done — overwrote table: {target_table} with {source_count} rows")

# ─────────────────────────────────────────────────────────────
# APPEND — insert source rows on top of existing data
# ─────────────────────────────────────────────────────────────
elif load_type == "APPEND":
    print(f"\n[APPEND] Strategy: Append all source records to target")
    print(f"\n[APPEND] Equivalent SQL:")
    print(f"  INSERT INTO {target_table}")
    print(f"  SELECT {', '.join(source_cols)}")
    print(f"  FROM source_data;")
    print(f"\n[APPEND] Executing append...")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"[APPEND] ✓ Done — appended {source_count} rows to: {target_table}")

# ─────────────────────────────────────────────────────────────
# MERGE — upsert: update existing rows, insert new rows
# ─────────────────────────────────────────────────────────────
elif load_type == "MERGE":
    keys           = [k.strip() for k in merge_key.split(",")]
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])

    # insert_timestamp preserved on update (keep original insert time)
    update_set     = {col: f"source.{col}" for col in source_cols if col != "insert_timestamp"}
    insert_set     = {col: f"source.{col}" for col in source_cols}

    update_clause  = ",\n        ".join([f"{k} = {v}" for k, v in update_set.items()])
    insert_col_str = ", ".join(insert_set.keys())
    insert_val_str = ", ".join(insert_set.values())

    print(f"\n[MERGE] Strategy   : Upsert — update matched rows, insert new rows")
    print(f"[MERGE] Keys       : {keys}")
    print(f"[MERGE] Condition  : {merge_condition}")
    print(f"[MERGE] Update cols: {list(update_set.keys())}")
    print(f"[MERGE] Insert cols: {list(insert_set.keys())}")
    print(f"\n[MERGE] Equivalent SQL:")
    print(f"  MERGE INTO {target_table} AS target")
    print(f"  USING source_data AS source")
    print(f"  ON {merge_condition}")
    print(f"  WHEN MATCHED THEN UPDATE SET")
    print(f"    {update_clause}")
    print(f"  WHEN NOT MATCHED THEN")
    print(f"    INSERT ({insert_col_str})")
    print(f"    VALUES ({insert_val_str});")

    if table_exists:
        print(f"\n[MERGE] Target table found — executing merge...")
        delta_table = DeltaTable.forName(spark, target_table)
        delta_table.alias("target") \
            .merge(updated_df.alias("source"), merge_condition) \
            .whenMatchedUpdate(set=update_set) \
            .whenNotMatchedInsert(values=insert_set) \
            .execute()
        print(f"[MERGE] ✓ Done — merged {source_count} source rows into: {target_table}")
    else:
        print(f"\n[MERGE] Target table not found — falling back to initial create...")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"[MERGE] ✓ Done — created table: {target_table} with {source_count} rows")

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected: FULL, APPEND, or MERGE.")

print("\n" + "=" * 65)
print("  LOAD COMPLETE")
print("=" * 65)

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))